In [1]:
from dotenv import load_dotenv
load_dotenv()

from openai import OpenAI
openai_client = OpenAI()

In [3]:
# Plain LLM

def llm(prompt):
    response = openai_client.responses.create(
        model="gpt-5.4-mini",
        input=prompt
    )
    return response.output_text

In [3]:
# testing the black box, where text comes in and text comes out

llm("Hey, what's up?")

'Hey! Not much—just here and ready to help. What’s going on?'

In [4]:
question = "I just discovered the course. Can I join now?"
answer = llm(question)
print(answer)

Yes — if the course is still open, you can usually join now.

If you want, I can help you figure out:
- whether enrollment is still open,
- how to join,
- or how to ask the instructor/admissions team.

If you meant a specific course, send me the course name or link and I’ll help you check.


In [ ]:
'''
The LLM gives a generic answer. It might say "you can usually join" or "check the course website." 
It doesn't know about our specific Zoomcamp courses, their enrollment policies, or their schedules. 
It tries to be helpful, but has no idea about actual enrollment status or policies.

This is different from a question like "how do I cook salmon?" - the LLM knows the answer because cooking salmon is common knowledge. 
But our courses are not in the training data.
'''

In [2]:
# Adding some context manually:

context = """
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we're still accepting submissions.

Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

What is the video/zoom link to the stream for the "Office Hours" or live/workshop sessions?
The zoom link is only published to instructors/presenters/TAs. Students participate via YouTube Live and submit questions to Slido.

Cloud alternatives with GPU
Check the quota and reset cycle carefully. Potential options include Google Colab, Kaggle, Databricks.
"""

In [3]:
# now buiding a prompt that takes into account the context

prompt = f"""
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."

Question:
{question}

Context:
{context}
"""

NameError: name 'question' is not defined

In [8]:
answer = llm(prompt)
print(answer)

Yes, you can still join now. If you want to receive a certificate, make sure to submit your project while submissions are still open.


In [ ]:
"""
RAG stands for Retrieval-Augmented Generation. Generation is the LLM producing text, and retrieval is search. 
We retrieve relevant documents from our knowledge base and use them to augment what the LLM generates. 
That search step is what gives the LLM the context it needs to answer correctly.

What we just did was naive. I knew in advance which FAQ entry held the answer and pasted it in by hand. 
What we want instead is to perform search automatically. We take the student's question, find the most relevant documents, and send those to the LLM.
"""

In [ ]:
# putting this idea in code, it will look something like:

def rag(question):
    search_results = search(question)
    user_prompt = build_prompt(question, search_results)
    return llm(user_prompt)

In [ ]:
"""
The LLM only sees the documents we hand it, so its answers are grounded in our data. If the right document is retrieved, the answer is good. 
If it's not, the LLM gets the wrong context and the answer is wrong. Your model is only as good as your retrieval, 
so search quality matters a lot for RAG.
"""

In [4]:
# The dataset for this exercise

import requests

docs_url = "https://datatalks.club/faq/json/courses.json"
response = requests.get(docs_url)
courses_raw = response.json()

#This returns a list of courses. Each course has a path field that points to its FAQ data.

In [5]:
# fecthing all FAQ documents from all courses:

documents = []
url_prefix = "https://datatalks.club/faq"

for course in courses_raw:
    course_url = f"""{url_prefix}{course["path"]}"""

    course_response = requests.get(course_url)
    course_response.raise_for_status()
    course_data = course_response.json()

    documents.extend(course_data)

len(documents)

1350

In [12]:
# looking at one entry:

documents[0]

{'id': '9e508f2212',
 'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: When does the course start?',
 'answer': "A new cohort runs roughly January–April every year. For the current cohort's exact start date and registration link, check the [course repo README](https://github.com/DataTalksClub/data-engineering-zoomcamp).\n\n- Register via the link in the course repo before the cohort starts.\n- Join the [course Telegram channel](https://t.me/dezoomcamp) for announcements.\n- Join DataTalks.Club's [Slack](https://datatalks.club/docs/general/slack/) and the `#course-data-engineering` channel."}

In [ ]:
# searching engine:

score = sim(query, document)

In [ ]:
"""
For each document in the database, you compute this score. Then you rank all documents by score and return the top N. 
What makes a search engine different from another search engine is what sim actually computes.

* text/lexical search (covered in this section): sim counts how many words the query and the document share. 
It looks at the surface form, the actual words, and matches them exactly.

* vector/semantic search (covered in module 2): sim compares the meaning of the query and the document. Same function, different similarity measure.
"""

In [ ]:
"""
Searching matters because we have around 1100 documents. Sending all of them to the LLM would be expensive and slow. 
The model would get confused with too much data. Search finds the most relevant documents to send instead.
"""

In [6]:
from minsearch import Index

index = Index(
    text_fields=["question", "section", "answer"],
    keyword_fields=["course"]
)

index.fit(documents)

In [7]:
question = "I just discovered the course. Can I join now?"

search_results = index.search(
    question,
    boost_dict={"question": 2.0, "section": 0.5},
    filter_dict={"course": "llm-zoomcamp"},
    num_results=5
)

search_results

"""
We used boost_dict to give the question field more weight (2.0 instead of the default 1.0) and section less weight (0.5). 
Query words appearing in the question field is a stronger signal than them appearing in the section name.
We used filter_dict to only return results from the LLM Zoomcamp course. Without this filter, we'd get results from all four courses.
"""

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '977bf7786c',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
  'answer': "You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date."},
 {'id': '69d122f12e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you c

In [8]:
[doc["question"] for doc in search_results]

['I just discovered the course. Can I still join?',
 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
 'How should I start the course and follow the weekly workflow?',
 'When will the course be offered next?']

In [10]:
# minsearch supports field boosting to reflect this fact that the question fields weight more than section fields:

results = index.search(
    question,
    num_results=5,
    boost_dict={"question": 2.0, "section": 0.5}
)

# This is the same boosting mechanism used by Elasticsearch and Lucene.

In [11]:
# Sometimes you want to restrict the search to a specific course. minsearch supports keyword filtering:

results = index.search(
    question,
    num_results=5,
    filter_dict={"course": "mlops-zoomcamp"}
)

In [12]:
[doc["question"] for doc in results]

['Course - Can I still join the course after the start date?',
 'Homework: Just found this course, can I still submit homeworks?',
 'I forgot if I registered, can I still join the zoomcamp?',
 'Certificate - Can I follow the course in a self-paced mode and get a certificate?',
 'Course: How do I start?']

In [14]:
# Wrapping it in a function:

def search(question, course="llm-zoomcamp"):
    boost_dict = {"question": 2.0, "section": 0.5}
    filter_dict = {"course": course}

    return index.search(
        question,
        boost_dict=boost_dict,
        filter_dict=filter_dict,
        num_results=5
    )

In [21]:
search_results = search(question)

TypeError: 'list' object is not callable

In [23]:
# Instructions (also called the system prompt): this tells the LLM how to behave. It never changes, so it's the same for every request.
# The instructions tell the LLM its role and how to answer:

INSTRUCTIONS = """
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."
"""

In [ ]:
USER_PROMPT_TEMPLATE = """
Question:
{question}

Context:
{context}
"""